## Excel Data Processing Utilities

This notebook contains helper functions for Excel data ingestion, column sanitization, and data type normalization. 
Designed for orchestrating Excel file processing pipelines in Databricks.

In [0]:
import pandas as pd
from contextlib import contextmanager
from datetime import datetime, timedelta

### Excel Column Index Conversion

Convert Excel column letters (A, B, ..., Z, AA, AB, etc.) to zero-based indices for pandas operations.

In [0]:
def excel_col_to_index(col):
    """
    Convert Excel column letter(s) to zero-based column index.
    Example: 'A' -> 0, 'Z' -> 25, 'AA' -> 26
    """
    # Normalize input to uppercase to handle both 'a' and 'A'
    col = col.upper()
    idx = 0
    
    # Iterate through each character in the column string
    # Formula: Base-26 conversion (A=1, B=2, ..., Z=26)
    for c in col:
        idx = idx * 26 + (ord(c) - ord("A") + 1)
    
    return idx - 1  # Convert to zero-based index for Python/pandas compatibility

### Column Name Sanitization

Clean and normalize column names to be database-friendly and SQL-compatible.
Handles special characters, whitespace, and common Excel formatting issues.

In [0]:
def sanitize_column(col: str) -> str:
    """
    Clean and normalize column names to be database-friendly:
    - Replace newlines with spaces
    - Convert to lowercase
    - Replace special characters (% -> _pct, µ -> um)
    - Keep only alphanumeric and underscores
    - Remove leading/trailing underscores
    """
    # Handle None values gracefully to prevent downstream errors
    if col is None:
        return ""
    
    # Convert to string to handle numeric column headers
    col = str(col)
    
    # Replace line breaks with spaces (common in merged Excel cells)
    col = col.replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
    
    # Normalize to lowercase for consistency
    col = col.lower()
    
    # Handle special characters commonly found in scientific/engineering data
    col = col.replace("%", "_pct")  # Percentage symbol
    col = col.replace("µ", "um")    # Micro symbol (e.g., micrometers)
    
    # Replace non-alphanumeric characters with underscore
    # This ensures SQL compatibility and prevents quoting issues
    col = re.sub(r"[^a-z0-9_]", "_", col)
    
    # Collapse multiple consecutive underscores into single one
    col = re.sub(r"_+", "_", col)
    
    # Remove leading/trailing underscores for cleaner names
    col = col.strip("_")

    return col

### Advanced Excel Range Parser

Parse Excel range strings (e.g., "A4:O36") with support for skipping specific rows and columns.
Returns parameters compatible with `pd.read_excel()`.

In [0]:
def parse_excel_range_advanced(
    table_range: str,
    skip_col_range: str = None,
    skip_row_range: str = None
):
    """
    Parse Excel range string (e.g., "A4:O36") and calculate:
    - Column indices to read (with skips)
    - Row indices to skip
    - Number of rows to read
    Returns a dictionary with parameters for pd.read_excel()
    """
    # Parse table range (example: A4:O36)
    start, end = table_range.split(":")

    # Extract column letters and row numbers using regex
    start_col = re.match(r"[A-Z]+", start).group()
    start_row = int(re.search(r"\d+", start).group())

    end_col = re.match(r"[A-Z]+", end).group()
    end_row = int(re.search(r"\d+", end).group())

    # Convert Excel column letters to zero-based indices
    start_col_idx = excel_col_to_index(start_col)
    end_col_idx = excel_col_to_index(end_col)

    # Generate list of column indices to read (inclusive range)
    usecols_idx = list(range(start_col_idx, end_col_idx + 1))

    # Handle column exclusions (e.g., skip merged header columns)
    if skip_col_range:
        sc1, sc2 = skip_col_range.split(":")
        skip_start = excel_col_to_index(sc1)
        skip_end = excel_col_to_index(sc2)

        # Filter out skipped columns from the range
        usecols_idx = [
            c for c in usecols_idx
            if not (skip_start <= c <= skip_end)
        ]

    # Build list of rows to skip (0-indexed for pandas)
    # Start with all rows before the table starts
    skiprows = list(range(start_row - 1))  # Convert 1-based Excel to 0-based pandas

    # Add any additional row ranges to skip (e.g., subtotal rows within data)
    if skip_row_range:
        r1, r2 = map(int, skip_row_range.split(":"))
        skiprows.extend(range(r1 - 1, r2))  # Convert to 0-indexed, inclusive

    # Calculate total number of rows to read (including skipped ones, pandas will handle skips)
    nrows = end_row - start_row + 1

    return {
        "usecols": usecols_idx,
        "skiprows": skiprows,
        "nrows": nrows,
        "start_row": start_row,
        "end_row": end_row,
    }

### Duplicate Column Name Resolution

Automatically rename duplicate column names by appending numeric suffixes (_2, _3, etc.).
Preserves all columns without data loss.

In [0]:
def fix_duplicate_columns(columns):
    """
    Makes column names unique by appending _2, _3, etc.
    Does NOT remove any column.
    Example: ['col', 'col', 'col'] -> ['col', 'col_2', 'col_3']
    """
    # Track occurrence count for each column name
    seen = defaultdict(int)
    new_cols = []

    for col in columns:
        seen[col] += 1
        
        # First occurrence keeps original name
        if seen[col] == 1:
            new_cols.append(col)
        else:
            # Subsequent occurrences get numeric suffix
            new_cols.append(f"{col}_{seen[col]}")

    return new_cols

### Data Type Normalization

Automatically detect and convert DataFrame column types:
1. Numeric conversion (handling commas and percentage signs)
2. Datetime detection and formatting
3. Fallback to string for remaining columns

In [0]:
def normalize_dataframe_types(
    df,
    datetime_format="%Y-%m-%d %H:%M:%S"
):
    """
    Automatically detect and convert data types:
    1. Try converting object columns to numeric (float)
    2. Try converting remaining object columns to datetime
    3. Convert anything else to string
    """
    # Work on copy to avoid modifying original DataFrame
    df = df.copy()

    # First pass: Convert to numeric where possible
    for col in df.columns:
        series = df[col]

        if series.dtype == object:
            # Clean common numeric formatting issues from Excel
            cleaned = (
                series.astype(str)
                .str.replace(",", "", regex=False)  # Remove thousand separators (e.g., "1,000" -> "1000")
                .str.replace("%", "", regex=False)  # Remove percentage signs for conversion
                .str.strip()
            )

            # Attempt conversion, coerce errors to NaN
            numeric = pd.to_numeric(cleaned, errors="coerce")

            # Only convert if at least some values are numeric (avoid converting pure text)
            if numeric.notna().sum() > 0:
                df[col] = numeric.astype(float)

    # Second pass: Convert remaining objects to datetime
    for col in df.columns:
        series = df[col]

        if series.dtype == object:
            # Try parsing as datetime with UTC handling
            parsed = pd.to_datetime(series, errors="coerce", utc=True)

            # Only convert if datetime parsing succeeded
            if parsed.notna().sum() > 0:
                df[col] = (
                    parsed
                    .dt.tz_convert(None)  # Remove timezone for consistent formatting
                    .dt.strftime(datetime_format)  # Format as string per specification
                )

    # Final pass: Ensure remaining columns are strings (not objects)
    for col in df.columns:
        if not (
            pd.api.types.is_float_dtype(df[col]) or
            pd.api.types.is_datetime64_any_dtype(df[col])
        ):
            df[col] = df[col].astype(str)

    return df

### Multi-Row Header Builder

Build hierarchical column headers from multi-row Excel headers with merged cells.
Handles forward-fill for merged cells and variable header depths.

In [0]:
def build_headers(df, header_rows=1, sep="_", normalize_newlines=True):
    """
    Build hierarchical column headers from multi-row Excel headers with merged cells.
    
    This function intelligently handles:
    - Horizontal merging (forward-filling first row values)
    - Variable header depth per column group
    - Special character normalization (e.g., newlines → spaces)
    
    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the header rows to process
    header_rows : int, optional
        Number of rows to consider for header building (default: 1)
    sep : str, optional
        Separator for concatenating header parts (default: "_")
    normalize_newlines : bool, optional
        Whether to convert newlines to spaces in values (default: True)
    
    Returns
    -------
    list
        List of compiled column headers
    """
    
    # Handle edge cases
    if df.empty or header_rows < 1:
        return []
    
    # -------------------------
    # SINGLE HEADER ROW CASE
    # -------------------------
    if header_rows == 1:
        final_headers = []
        unnamed_count = 1

        for col_idx, val in enumerate(df.iloc[0]):
            # Handle empty cells with default naming
            if pd.isna(val) or str(val).strip() == "":
                final_headers.append(f"unnamed_col_{unnamed_count}")
                unnamed_count += 1
                continue

            # Clean and normalize the header value
            cleaned_value = str(val).strip()

            if normalize_newlines:
                cleaned_value = cleaned_value.replace("\n", " ")
                cleaned_value = " ".join(cleaned_value.split())  # Remove extra spaces

            final_headers.append(cleaned_value)

        # Final normalization: lowercase, spaces to underscores, handle %
        return [
            h.lower()
             .replace(" ", "_")
             .replace("%", "pct")
            for h in final_headers
        ]
    
    # -------------------------
    # MULTI-ROW HEADER CASE
    # -------------------------
    # Ensure we don't exceed available rows
    header_rows = min(header_rows, df.shape[0])
    
    # Create copy to avoid modifying original
    header_data = df.iloc[:header_rows].copy()
    
    # Forward-fill first row to handle horizontally merged cells
    # This propagates merged header values across all affected columns
    header_data.iloc[0] = header_data.iloc[0].ffill()
    
    final_headers = []
    
    # Process each column individually
    for col_idx in range(df.shape[1]):
        header_parts = []
        
        # Collect non-empty values from each header row
        for row_idx in range(header_rows):
            cell_value = header_data.iloc[row_idx, col_idx]
            
            # Skip NaN/empty values
            if pd.isna(cell_value) or str(cell_value).strip() == "":
                continue
            
            # Clean the value
            cleaned_value = str(cell_value).strip()
            
            if normalize_newlines:
                cleaned_value = cleaned_value.replace('\n', ' ')
                cleaned_value = " ".join(cleaned_value.split())  # Normalize whitespace
            
            header_parts.append(cleaned_value)
        
        # Join parts with separator or use default name
        if header_parts:
            final_header = sep.join(header_parts)
            final_headers.append(final_header)
        else:
            # Fallback to default column name if all values are empty
            final_headers.append(f"col_{col_idx}")
    
    # Final normalization
    final_headers = [
        h.lower()
        .replace(" ", "_")
        .replace("%", "pct")
        for h in final_headers
    ]

    return final_headers

### Blank Column Handler

Rename columns where the header is blank, None, NaN, or whitespace-only
to standardized names ('Unnamed column 1', 'Unnamed column 2', etc.).

In [0]:
def fill_blank_columns(df):
    """
    Renames columns where the column name is blank, None, NaN, or only whitespace
    to 'Unnamed column 1', 'Unnamed column 2', etc.
    """
    df = df.copy()

    new_columns = []
    unnamed_count = 1

    for col in df.columns:
        # Check for various forms of "empty" column names
        if (
            col is None
            or (isinstance(col, float) and pd.isna(col))
            or str(col).strip() == ""
        ):
            new_columns.append(f"Unnamed column {unnamed_count}")
            unnamed_count += 1
        else:
            new_columns.append(col)

    df.columns = new_columns
    return df

### 🧹 Empty Row Removal

Remove rows where all values are NaN, empty strings, or "nan" strings.
Common cleanup step after reading Excel files with trailing empty rows

In [0]:
def drop_all_nan_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove rows where all values are NaN.
    """
    # Normalize various representations of missing values to actual NaN
    df = df.replace(["nan", "NaN", ""], np.nan)
    
    # Drop rows that are entirely NaN (how='all' requires all values to be NaN)
    return df.dropna(how="all")

### 🔗 URL-Friendly String Slugification

Convert strings to ASCII slugs suitable for URLs, filenames, or identifiers.
Removes special characters and normalizes whitespace.

In [0]:
def slugify(value, allow_unicode=False):
    """
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    
    # Handle Unicode normalization
    if allow_unicode:
        value = unicodedata.normalize('NFKC', value)
    else:
        # Convert to ASCII, ignoring non-ASCII characters
        value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    
    # Remove special characters, keep only word chars, spaces, and hyphens
    value = re.sub(r'[^\w\s-]', '', value.lower())
    
    # Replace spaces and repeated dashes with single dash
    return re.sub(r'[-\s]+', '-', value).strip('-_')

### 🗄️ Delta Lake Existence Check

Verify if a Delta table exists at the specified path without raising exceptions.
Useful for conditional logic in ETL pipelines (merge vs. create).

In [0]:
def delta_exists(spark, path):
    """
    Check if a Delta table exists at the given path.
    Returns True if exists, False otherwise.
    """
    try:
        # Attempt to create DeltaTable object for the path
        DeltaTable.forPath(spark, path)
        return True
    except AnalysisException:
        # Table doesn't exist or path is invalid
        return False

### 📅 Dynamic Datetime Token Replacement

Replace datetime tokens in strings with current or specified datetime values.
Supports partitioning patterns and filename generation.

In [0]:
def apply_datetime_tokens(text: str, dt: datetime = None) -> str:
    """
    Replaces datetime tokens in a string using replace().
    
    Supported tokens:
      {yyyy}          -> 4-digit year
      {MM}            -> 2-digit month
      {dd}            -> 2-digit day
      {HH}            -> 2-digit hour (24h)
      {yyyyMMddHH}    -> full timestamp
    
    Example:
      reportediarioplantas/{yyyy}/{MM}/{dd}/{HH}
      Base_Datos_Reporte_Diario_Plantas_2025GIG_{yyyyMMddHH}.xlsx
    """
    # Default to current UTC time if not specified
    if dt is None:
        dt = datetime.utcnow()

    # Define token mappings
    replacements = {
        "{yyyy}": dt.strftime("%Y"),
        "{MM}": dt.strftime("%m"),
        "{dd}": dt.strftime("%d"),
        "{HH}": dt.strftime("%H"),
        "{yyyyMMddHH}": dt.strftime("%Y%m%d%H"),
    }

    # Replace all tokens in the text
    for token, value in replacements.items():
        text = text.replace(token, value)

    return text

### JDBC Connection Context Manager

Safe context manager for JDBC database connections using Spark's JVM gateway.
Ensures connections are properly closed even if exceptions occur.

In [0]:
@contextmanager
def jdbc_connection(jdbc_url):
    """
    Context manager for JDBC connections via Spark JVM gateway.
    Automatically closes connection on exit.
    """
    conn = None
    try:
        # Create connection using Spark's JVM gateway
        conn = spark._sc._gateway.jvm.java.sql.DriverManager.getConnection(jdbc_url)
        yield conn
    finally:
        # Ensure connection is closed to prevent resource leaks
        if conn:
            conn.close()

### Value Adjustment Utility

Subtract a specified amount from integer or datetime values.
Useful for date arithmetic (e.g., "previous day") or numeric adjustments.

In [0]:
def adjust_value(value: str, substract: str):
    """
    Subtract value from either an integer or datetime.
    Attempts integer conversion first, falls back to datetime.
    """
    # Try integer conversion first
    try:
        num = int(value)
        return num - int(substract)
    except ValueError:
        pass

    # Try datetime (ISO-like formats) if not an integer
    try:
        dt = datetime.fromisoformat(value)
        # Subtract specified number of days
        return dt - timedelta(days=int(substract))
    except ValueError:
        pass

    # Raise error if neither format matches
    raise ValueError("String is neither a valid int nor a valid datetime")
